# Pairwise judging (Task 7)

Run this notebook on Kaggle **as a utility benchmark notebook with model access enabled** (the `kaggle_benchmarks` model proxy). Attach the dataset made from `judge_bundle.zip` (it holds `scripts/`, `src/` and `outputs/eval/*_pairs.jsonl`). No model outputs are regenerated here: the judge reads the saved pairwise JSONL files only.

The notebook is resumable: re-running it skips pairs that already have judgments in `/kaggle/working/pa2/outputs/eval/`.

In [ ]:
import glob, os, shutil, subprocess, sys
SRC = glob.glob('/kaggle/input/**/judge_bundle', recursive=True) or glob.glob('/kaggle/input/*', recursive=False)
print(SRC)
BUNDLE = next(p for p in SRC if os.path.exists(os.path.join(p, 'scripts', 'judge_pairwise_kaggle.py')))
WORK = '/kaggle/working/pa2'
if not os.path.exists(WORK):
    shutil.copytree(BUNDLE, WORK)
os.chdir(WORK)
print(sorted(glob.glob('outputs/eval/*_pairs.jsonl')))

In [ ]:
# Check which judge models the proxy exposes; keep google/gemini-3.6-flash if listed.
import kaggle_benchmarks as kbench
try:
    print(sorted(kbench.llms.keys()) if hasattr(kbench.llms, 'keys') else kbench.llms)
except Exception as e:
    print('could not list models:', e)
JUDGE = 'google/gemini-3.6-flash'

In [ ]:
# Smoke test: one tiny request through the proxy
from importlib import util
spec = util.spec_from_file_location('judge', 'scripts/judge_pairwise_kaggle.py'); judge = util.module_from_spec(spec); spec.loader.exec_module(judge)
print(judge.KaggleJudge(JUDGE)('Reply with exactly: [{"item": 1, "reason": "ok", "verdict": "tie"}]'))

In [ ]:
for pairs in sorted(glob.glob('outputs/eval/*_pairs.jsonl')):
    out = pairs.replace('_pairs.jsonl', '_judged.jsonl')
    print('=' * 30, pairs)
    subprocess.run([sys.executable, 'scripts/judge_pairwise_kaggle.py', '--pairs_path', pairs, '--output_path', out,
                    '--judge_model', JUDGE, '--batch_size', '5', '--sleep_seconds', '5', '--max_retries', '3',
                    '--reuse_judgments'], check=False)

In [ ]:
import json
for s in sorted(glob.glob('outputs/eval/*_judged_summary.json')):
    print(s, json.dumps(json.load(open(s)), indent=1))
shutil.make_archive('/kaggle/working/judgments', 'zip', 'outputs/eval')
print('Download /kaggle/working/judgments.zip (or Save Version) and send it back.')